## 介紹 

本課程將涵蓋： 
- 甚麼是函式呼叫及其使用案例 
- 如何使用 Azure OpenAI 建立函式呼叫 
- 如何將函式呼叫整合到應用程式中 

## 學習目標 

完成本課程後，您將知道如何做並理解： 

- 使用函式呼叫的目的 
- 使用 Azure Open AI 服務設定函式呼叫 
- 為您的應用程式使用案例設計有效的函式呼叫 


## 理解函數調用 

在本課程中，我們想為我們的教育初創企業建立一個功能，讓用戶可以使用聊天機器人來尋找技術課程。我們將推薦符合他們技能水平、當前職位和感興趣技術的課程。 

為此，我們將使用組合： 
 - `Azure Open AI` 為用戶創建聊天體驗
 - `Microsoft Learn Catalog API` 幫助用戶根據其請求找到課程 
 - `Function Calling` 將用戶的查詢發送給函數以進行 API 請求。 

為了開始，讓我們來看看為什麼我們首先要使用函數調用： 


### 為什麼要使用函數呼叫 

如果你已經完成本課程中其他課程，你大概已經了解使用大型語言模型 (LLMs) 的威力。希望你也能看到它們的一些限制。 

函數呼叫是 Azure Open AI 服務的一個功能，用來克服以下限制： 
1) 回應格式的一致性 
2) 能夠在聊天上下文中使用應用程式其他來源的數據 

在函數呼叫之前，LLM 的回應是無結構且不一致的。開發者必須撰寫複雜的驗證程式碼，以確保能處理每種回應的變化。 

使用者無法得到像「斯德哥爾摩當前的天氣如何？」這樣的答案。這是因為模型的資料僅限於訓練時的時間點。 

我們來看下面的範例說明這個問題： 

假設我們想建立一個學生資料庫，以便向他們推薦合適的課程。以下有兩個學生的描述，其中包含的數據非常相似。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我們想將此發送給大型語言模型 (LLM) 來解析數據。這可以在我們的應用程式中用來發送至 API 或存儲到資料庫中。

讓我們創建兩個相同的提示，指示 LLM 我們感興趣的資訊：


我們想將這個發送給大型語言模型（LLM）來解析對我們產品重要的部分。這樣我們可以創建兩個相同的提示來指示LLM：


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


建立這兩個提示後，我們會使用 `client.responses.create` 將它們發送到 LLM。我們將提示存儲在 `input` 變量中，並將角色指定為 `user`。這是為了模擬用戶向聊天機械人發送訊息的情況。



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

現在我們可以同時發送這兩個請求到 LLM，並檢視我們收到的回應。 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



儘管提示相同且描述相似，但我們可以獲得不同格式的 `Grades` 屬性。

如果多次執行上述儲存格，格式可能是 `3.7` 或 `3.7 GPA`。

這是因為 LLM 接收的提示是非結構化的文字資料，並且回傳的也是非結構化資料。我們需要有結構化的格式，才能在儲存或使用這些資料時知道預期的內容。

透過使用函式呼叫，我們可以確保收到結構化資料。使用函式呼叫時，LLM 並不會實際呼叫或執行任何函式，而是根據我們設計的結構來回應。我們再利用這些結構化回應來決定在應用程式中該呼叫哪個函式。
 


![函數呼叫流程圖](../../../../translated_images/zh-MO/Function-Flow.083875364af4f4bb.webp)


然後，我們可以將從函數返回的結果發送回大型語言模型（LLM）。LLM 將使用自然語言回應，以回答用戶的查詢。


### 使用函數調用的案例

<strong>呼叫外部工具</strong>
聊天機械人擅長為用戶提供問題答案。通過使用函數調用，聊天機械人可利用用戶訊息來完成某些任務。例如，學生可以請聊天機械人「發送電郵給我的導師，說我需要更多這科的幫助」。這可以調用名為 `send_email(to: string, body: string)` 的函數。


**建立 API 或資料庫查詢**
用戶可使用自然語言查找信息，並將其轉換成格式化的查詢或 API 請求。例如老師可以請求「誰完成了上一次作業」，這可以調用名為 `get_completed(student_name: string, assignment: int, current_status: string)` 的函數。


<strong>建立結構化數據</strong>
用戶可以將一段文字或 CSV，利用大型語言模型(Large Language Model, LLM)提取重要信息。例如學生可以將關於和平協議的維基百科文章轉換成 AI 問題卡。這可通過調用名為 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 的函數來完成。


## 2. 建立你的第一個函數呼叫

建立函數呼叫的過程包括三個主要步驟：
1. 使用你的函數列表和用戶訊息呼叫 Chat Completions API
2. 讀取模型的回應以執行動作，例如執行函數或 API 呼叫
3. 使用函數的回應，再次呼叫 Chat Completions API，利用該資訊建立對用戶的回應。


![函數呼叫流程](../../../../translated_images/zh-MO/LLM-Flow.3285ed8caf4796d7.webp)


### 函數調用的元素

#### 使用者輸入

第一步是建立一條使用者訊息。這可以透過取得文字輸入的值來動態指定，或者你可以在這裡指定一個值。如果你是第一次使用 Chat Completions API，我們需要定義訊息的 `role` 和 `content`。

`role` 可以是 `system`（建立規則）、`assistant`（模型）或 `user`（最終使用者）。對於函數調用，我們會將其指定為 `user` 並給出範例問題。


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 建立函數。

接著我們將定義一個函數及該函數的參數。我們這裡只會使用一個函數，稱為 `search_courses`，但你可以建立多個函數。

<strong>重要</strong>：函數會包含在發送給 LLM 的系統訊息中，並會計入你可用的 token 數量內。


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

`name` - 我們想要呼叫的函數名稱。 

`description` - 這是函數如何運作的描述。在這裡，明確和清晰是很重要的 

`parameters` - 你希望模型在回應中產生的值和格式列表 


`type` - 屬性將被儲存的資料類型 

`properties` - 模型將用於回應的具體值列表 


`name` - 模型在格式化回應中會使用的屬性名稱 

`type` - 該屬性的資料類型 

`description` - 該具體屬性的描述 


<strong>選用</strong>

`required` - 函數呼叫要完成所需的必要屬性 


### 呼叫函數
定義函數後，我們現在需要在呼叫 Chat Completion API 時包含該函數。我們透過在請求中加入 `functions` 來做到這一點。在這個例子中是 `functions=functions`。

也有一個選項可以將 `function_call` 設為 `auto`。這表示我們會讓大型語言模型根據用戶訊息決定應該呼叫哪個函數，而不是我們自己指定。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


現在讓我們來看看回應，了解它是如何格式化的：

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以看到函數名稱被呼叫，並且從使用者訊息中，LLM 能夠找到符合函數參數的資料。


## 3.將函數調用整合到應用程式中。


在我們測試了來自大型語言模型的格式化回應後，現在可以將其整合到應用程式中。

### 管理流程

要將此整合到我們的應用程式中，請按照以下步驟進行：

首先，呼叫 Open AI 服務並將訊息存入名為 `response_message` 的變量。


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


現在我們將定義一個函式，該函式會呼叫 Microsoft Learn API 以獲取課程列表： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作為一項最佳實踐，我們將查看模型是否想呼叫一個函數。然後，我們將創建一個可用的函數，並將其與正在呼叫的函數匹配。 
然後，我們將函數的參數對應到來自 LLM 的參數。

最後，我們將附加函數呼叫訊息和由 `search_courses` 訊息返回的值。這為 LLM 提供了所有回應使用者所需的自然語言資訊。


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


現在我們將把更新後的訊息發送到 LLM，這樣我們就能收到自然語言的回應，而不是 API JSON 格式的回應。 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## 代碼挑戰 

做得好！為繼續學習 Azure Open AI 函數呼叫，你可以建立：https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 更多函數參數，可能可以幫助學習者找到更多課程。你可以在這裡找到可用的 API 參數： 
 - 建立另一個函數呼叫，收集學習者更多資訊，例如他們的母語 
 - 建立錯誤處理，當函數呼叫和／或 API 呼叫沒有回傳合適課程時 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
